## Part 1 - Data Collection

### Import Libraries

In [1]:
import polars as pl

In [3]:
# Set the streaming engine as the default for all future operations
pl.Config.set_engine_affinity("streaming")
# Show all rows
pl.Config.set_tbl_rows(-1)

polars.config.Config

### Load Data

In [4]:
transaction_data = pl.scan_parquet(
    "../data/filtered_transactions.parquet"
)

In [5]:
# Rename the Account and Account_duplicated_0 columns based on Kaggle Source page
transaction_data = transaction_data.rename({
    "Account": "From Account",
    "Account_duplicated_0": "To Account"
})

# Get the column names from the lazy schema
columns = transaction_data.collect_schema().names()

transaction_data.collect_schema()

Schema([('Timestamp', Datetime(time_unit='us', time_zone=None)),
        ('From Bank', Int64),
        ('From Account', String),
        ('To Bank', Int64),
        ('To Account', String),
        ('Amount Received', Float64),
        ('Receiving Currency', String),
        ('Amount Paid', Float64),
        ('Payment Currency', String),
        ('Payment Format', String),
        ('Is Laundering', Int64)])

In [6]:
transaction_data.head(5).collect()

Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
datetime[μs],i64,str,i64,str,f64,str,f64,str,str,i64
2022-08-01 00:15:00,20,"""800104D70""",20,"""800104D70""",8095.07,"""US Dollar""",8095.07,"""US Dollar""","""Reinvestment""",0
2022-08-01 00:18:00,3196,"""800107150""",3196,"""800107150""",7739.29,"""US Dollar""",7739.29,"""US Dollar""","""Reinvestment""",0
2022-08-01 00:23:00,1208,"""80010E430""",1208,"""80010E430""",2654.22,"""US Dollar""",2654.22,"""US Dollar""","""Reinvestment""",0
2022-08-01 00:19:00,3203,"""80010EA80""",3203,"""80010EA80""",13284.41,"""US Dollar""",13284.41,"""US Dollar""","""Reinvestment""",0
2022-08-01 00:27:00,20,"""800104D20""",20,"""800104D20""",9.72,"""US Dollar""",9.72,"""US Dollar""","""Reinvestment""",0


In [7]:
# Number of rows
transaction_data.select(
    pl.len().alias("num_transactions")
).collect()

num_transactions
u32
176060273


In [8]:
# Check unique values for "Is Laundering"
transaction_data.select(
    "Is Laundering"
).unique().collect()

Is Laundering
i64
0
1


### Data Preprocessing

In [9]:
transaction_data.select(
    pl.col("Timestamp").min().alias("start_date"),
    pl.col("Timestamp").max().alias("end_date")
).collect()

start_date,end_date
datetime[μs],datetime[μs]
2022-08-01 00:00:00,2022-11-05 23:59:00


#### Add a column to distinguish if its an internal or external transfer

In [10]:
transaction_data = transaction_data.with_columns(
    pl.when(pl.col("From Account") == pl.col("To Account"))
    .then(pl.lit("Self-Transfer"))
    .when(pl.col("From Bank") != pl.col("To Bank"))
    .then(pl.lit("Cross-Bank Transfer"))
    .otherwise(pl.lit("Standard Transfer"))
    .alias("Transfer Type")
)

#### Missing Values

In [11]:
# Count missing values in every column
missing_values = transaction_data.select(
    [
        pl.col(column).null_count().alias(column)
        for column in columns
    ]
).collect()

missing_values

Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0


#### Duplicates

In [12]:
# Narrow down duplicate search by first doing a check on a few columns

candidate_key = [
    "Timestamp",
    "From Account",
    "To Account",
    "Amount Received",
    "Payment Format"
]

duplicate_timestamps = (
    transaction_data
    .group_by(candidate_key)
    .agg(pl.len().alias("timestamp_count"))
    .filter(pl.col("timestamp_count") > 1)
    .select("Timestamp")
)

# Keep only rows having a repeated timestamp
candidate_rows = transaction_data.join(
    duplicate_timestamps,
    on="Timestamp",
    how="semi",
)

# Then find duplicates on the smaller subset - required because data is large
# Check exact duplicates only among candidates
duplicate_rows = (
    candidate_rows
    .group_by(columns)
    .agg(pl.len().alias("group_count"))
    .filter(pl.col("group_count") > 1)
    .select(
        pl.col("group_count")
        .sum()
        .alias("rows_in_duplicate_groups")
    )
    .collect(engine="streaming")
)

duplicate_rows

rows_in_duplicate_groups
u32
328


In [13]:
deduplicated_candidates = candidate_rows.unique(
    subset=columns,
    keep="first",
    maintain_order=False,
)

# Rows with timestamps that never had a candidate duplicate
non_candidate_rows = transaction_data.join(
    duplicate_timestamps,
    on="Timestamp",
    how="anti",
)

# Recombine the cleaned candidate rows and unaffected rows
cleaned_transactions = pl.concat(
    [non_candidate_rows, deduplicated_candidates]
)

In [14]:
cleaned_transactions.select(
    pl.len().alias("num_transactions")
).collect()

num_transactions
u32
176060109


#### Check categories of categorical columns

In [15]:
categorical_columns = [
    "Receiving Currency",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]

for column in categorical_columns:
    print(column)
    print(
        cleaned_transactions
        .group_by(column)
        .len()
        .sort("len", descending=True)
        .collect()
    )

Receiving Currency
shape: (15, 2)
┌────────────────────┬──────────┐
│ Receiving Currency ┆ len      │
│ ---                ┆ ---      │
│ str                ┆ u32      │
╞════════════════════╪══════════╡
│ US Dollar          ┆ 66202744 │
│ Euro               ┆ 41558467 │
│ Yuan               ┆ 9295974  │
│ Rupee              ┆ 6980256  │
│ Canadian Dollar    ┆ 5732738  │
│ Swiss Franc        ┆ 5632441  │
│ Australian Dollar  ┆ 5497144  │
│ UK Pound           ┆ 5356000  │
│ Yen                ┆ 5098728  │
│ Brazil Real        ┆ 4984293  │
│ Ruble              ┆ 4641082  │
│ Mexican Peso       ┆ 4124790  │
│ Saudi Riyal        ┆ 3701014  │
│ Bitcoin            ┆ 3638166  │
│ Shekel             ┆ 3616272  │
└────────────────────┴──────────┘
Payment Currency
shape: (15, 2)
┌───────────────────┬──────────┐
│ Payment Currency  ┆ len      │
│ ---               ┆ ---      │
│ str               ┆ u32      │
╞═══════════════════╪══════════╡
│ US Dollar         ┆ 66696768 │
│ Euro              ┆ 

### Save Data

In [16]:
# Save the lazy cleaned dataset directly to a parquet (heavy compression)
cleaned_transactions.sink_parquet("../data/filtered_cleaned_transactions.parquet")